# GasOpt: synthetic model validation

**MSc Finance & Big Data — Stochastic Optimization of Ethereum Transaction Fees under Gas-Price Uncertainty**

**SYNTHETIC PHASE 1 DATA. These inputs are not empirical Ethereum observations.**

Audience: an optimization-course reader familiar with expectations and linear
programming. Prerequisites: Python 3.12+, the installed `gasopt` package and the
`Python (GasOpt)` kernel (see README). Run all cells from top to bottom.

We will derive the model, audit its units and feasibility, prove a risk-neutral
equivalence, and verify the risk-aversion experiment independently of the solver.
No external data is downloaded by this notebook.

## 1. Problem definition

**How should a crypto treasury schedule delay-tolerant Ethereum transactions to
minimize expected transaction costs while controlling exposure to extreme gas-fee scenarios?**

All five transactions are available at slot 1. Each must execute once by its
inclusive deadline. We choose a single schedule *before* knowing which complete
four-slot price trajectory occurs. Decisions cannot depend on the realized
scenario: this is a static, here-and-now model with no recourse.

Slot durations are unspecified business time buckets in Phase 1. Synthetic
prices represent an all-in price per gas; gas requirements are fixed. Optional
transaction-count capacities are supported but disabled in the main experiment.

In [1]:
from importlib.metadata import version
from itertools import product
from math import isclose
import platform

import numpy as np
import pandas as pd
import pyomo.environ as pyo
from IPython.display import display

from gasopt.data.synthetic import toy_data
from gasopt.evaluation.metrics import mean_prices_gwei, scenario_costs_eth, weighted_var_cvar
from gasopt.experiment import sensitivity_table
from gasopt.models import solve_deterministic, solve_stochastic
from gasopt.models.deterministic import build_deterministic_model
from gasopt.models.stochastic_cvar import build_stochastic_model
from gasopt.types import OptimizationConfig

pd.set_option('display.float_format', lambda value: f'{value:.9f}')
transactions, scenarios = toy_data()
alpha = 0.80
display(pd.Series({'Python': platform.python_version(), **{
    name: version(name) for name in ('gasopt', 'pyomo', 'highspy', 'numpy', 'pandas')
}}, name='Executed environment').to_frame())

,Executed environment
Python,3.12.2
gasopt,0.1.0
pyomo,6.10.1
highspy,1.15.1
numpy,2.5.3
pandas,3.0.6


## 2. Synthetic transactions

$g_i$ is the transaction's gas requirement and $d_i$ its latest permissible slot.
These gas values are assumptions supplied for the toy experiment, including the
familiar 21,000 value; they have not been sampled from Ethereum.

In [2]:
transaction_table = pd.DataFrame([
    {'transaction': tx.id, 'gas_used': tx.gas_used, 'deadline_slot': tx.deadline,
     'admissible_slots': tuple(range(1, tx.deadline + 1))}
    for tx in transactions
]).set_index('transaction')
display(transaction_table)
print(f'Total synthetic gas requirement: {sum(tx.gas_used for tx in transactions):,} gas')

,gas_used,deadline_slot,admissible_slots
transaction,,,
tx1,21000,1,"(1,)"
tx2,65000,2,"(1, 2)"
tx3,120000,4,"(1, 2, 3, 4)"
tx4,48000,3,"(1, 2, 3)"
tx5,180000,4,"(1, 2, 3, 4)"


Total synthetic gas requirement: 434,000 gas


## 3. Synthetic gas-price scenarios

Each row is a complete trajectory, preserving the supplied joint relationship
across slots. We do not draw independent slot prices or regenerate these values.
Initially $q_s=1/5$. Prices are in **gwei/gas**.

In [3]:
price_table = pd.DataFrame([s.prices_gwei for s in scenarios],
    index=[s.id for s in scenarios], columns=[f'slot_{t}' for t in range(1, 5)])
probabilities = np.array([s.probability for s in scenarios])
display(price_table.assign(probability=probabilities))
means = np.array(mean_prices_gwei(scenarios))
display(pd.DataFrame({'weighted_mean_gwei_per_gas': means}, index=price_table.columns))
assert np.allclose(means, [12.8, 11.4, 8.8, 10.6])

,slot_1,slot_2,slot_3,slot_4,probability
s1,12,9,7,11,0.200000000
s2,10,14,8,6,0.200000000
s3,20,18,11,9,0.200000000
s4,7,6,10,15,0.200000000
s5,15,10,8,12,0.200000000


,weighted_mean_gwei_per_gas
slot_1,12.800000000
slot_2,11.400000000
slot_3,8.800000000
slot_4,10.600000000


## 4. Mathematical notation

| Symbol | Meaning | Unit / domain |
|:--|:--|:--|
| $i\in I$, $t\in T$, $s\in S$ | Transaction, slot, scenario | Indices |
| $g_i$ | Gas requirement | gas |
| $d_i$ | Inclusive deadline | Slot index |
| $p_{ts}$ | Price in slot $t$, scenario $s$ | gwei/gas |
| $q_s$ | Scenario probability, $\sum_s q_s=1$ | Dimensionless |
| $K_t$ | Optional slot capacity | Transaction count |
| $\alpha$ | CVaR confidence level | $(0,1)$ |
| $\lambda$ | Nonnegative risk-aversion weight | Dimensionless |
| $x_{it}$ | 1 if transaction $i$ executes in slot $t$ | Binary |
| $\eta$ | CVaR threshold / VaR proxy | ETH, unrestricted |
| $\xi_s$ | Positive scenario loss above $\eta$ | ETH, nonnegative |

Let $A=\{(i,t):1\le t\le d_i\}$. The model creates binaries only for $A$:

$$\sum_{t:(i,t)\in A}x_{it}=1\quad\forall i,\qquad x_{it}\in\{0,1\}.$$

Exactly-one assignment prevents omission or duplication; excluding $t>d_i$
enforces deadlines. Optional capacity is
$\sum_{i:(i,t)\in A}x_{it}\le K_t$. Results pad excluded pairs with zeros.

**Unit audit:**
$$\underbrace{g_i}_{\mathrm{gas}}\,
  \underbrace{p_{ts}}_{\mathrm{gwei/gas}}\,
  \underbrace{10^{-9}}_{\mathrm{ETH/gwei}}
  =\mathrm{ETH},\qquad 1\ \mathrm{ETH}=10^9\ \mathrm{gwei}.$$

For example, 21,000 gas at 12 gwei/gas costs 252,000 gwei = **0.000252 ETH**.
All optimization cost expressions and auxiliary cost variables use ETH.

**Phase 2 extension (backward compatible):** transactions may now have a release slot $r_i$. The admissible set becomes $A=\{(i,t):r_i\leq t\leq d_i\}$ and $\sum_{t=r_i}^{d_i}x_{it}=1$. Omitted releases default to $r_i=1$, so all Phase 1 calculations and results in this notebook remain unchanged. Release windows and priority classes are business assumptions.

## 5. Deterministic optimization

Compute $\bar p_t=\sum_s q_s p_{ts}$ and solve

$$\min_x\ 10^{-9}\sum_{(i,t)\in A}g_i\bar p_t x_{it}.$$

Without capacities this separates by transaction: choose its cheapest admissible
mean-price slot. The table below contains objective coefficients in ETH; missing
entries are forbidden decisions.

In [4]:
coefficients = pd.DataFrame([
    [tx.gas_used * means[t - 1] * 1e-9 if t <= tx.deadline else np.nan
     for t in range(1, 5)] for tx in transactions
], index=[tx.id for tx in transactions], columns=price_table.columns)
display(coefficients)
deterministic = solve_deterministic(transactions, scenarios)
assignment = pd.DataFrame([
    [deterministic.x_values[tx.id, t] for t in range(1, 5)] for tx in transactions
], index=[tx.id for tx in transactions], columns=price_table.columns)
display(assignment)
manual_expected = (21_000 * 12.8 + 65_000 * 11.4 + (120_000 + 48_000 + 180_000) * 8.8) * 1e-9
print('Schedule:', deterministic.schedule)
print(f'Manual expected cost: {manual_expected:.9f} ETH')
print(f'Solver objective:    {deterministic.objective_value_eth:.9f} ETH')
assert isclose(manual_expected, deterministic.objective_value_eth, abs_tol=1e-12)
assert np.allclose(assignment.sum(axis=1), 1)
assert all(deterministic.schedule[tx.id] <= tx.deadline for tx in transactions)

,slot_1,slot_2,slot_3,slot_4
tx1,0.000268800,NaN,NaN,NaN
tx2,0.000832000,0.000741000,NaN,NaN
tx3,0.001536000,0.001368000,0.001056000,0.001272000
tx4,0.000614400,0.000547200,0.000422400,NaN
tx5,0.002304000,0.002052000,0.001584000,0.001908000


,slot_1,slot_2,slot_3,slot_4
tx1,1.000000000,0.000000000,0.000000000,0.000000000
tx2,0.000000000,1.000000000,0.000000000,0.000000000
tx3,0.000000000,0.000000000,1.000000000,0.000000000
tx4,0.000000000,0.000000000,1.000000000,0.000000000
tx5,0.000000000,0.000000000,1.000000000,0.000000000


Schedule: {'tx1': 1, 'tx2': 2, 'tx3': 3, 'tx4': 3, 'tx5': 3}
Manual expected cost: 0.004072200 ETH
Solver objective:    0.004072200 ETH


The optimum is $(1,2,3,3,3)$. Transaction 1 must execute in slot 1; transaction 2
cannot wait for slot 3. The other transactions use slot 3. The expected bill is

$$0.0002688+0.0007410+0.0030624=\boxed{0.0040722\ \mathrm{ETH}}.$$

## 6. Expected-cost stochastic formulation

Now retain each scenario explicitly:

$$C_s(x)=10^{-9}\sum_{(i,t)\in A}g_i p_{ts}x_{it},\qquad
\min_x E[C(x)]=\min_x\sum_s q_s C_s(x).$$

The table below breaks every scenario cost into five transaction contributions.
It then shows the multiplication by $q_s$ used to calculate the expectation.

In [5]:
expected_value = solve_stochastic(transactions, scenarios, OptimizationConfig(lambda_risk=0))
contributions = pd.DataFrame({tx.id: [
    tx.gas_used * s.prices_gwei[expected_value.schedule[tx.id] - 1] * 1e-9
    for s in scenarios] for tx in transactions}, index=[s.id for s in scenarios])
scenario_audit = contributions.assign(
    total_eth=contributions.sum(axis=1), probability=probabilities,
    weighted_cost_eth=contributions.sum(axis=1) * probabilities)
display(scenario_audit)
print(f'Sum of weighted costs: {scenario_audit.weighted_cost_eth.sum():.9f} ETH')
assert np.allclose(scenario_audit.total_eth,
    [.003273, .003904, .005418, .004017, .003749], atol=1e-12, rtol=0)
assert isclose(expected_value.objective_value_eth, manual_expected, abs_tol=1e-12)

,tx1,tx2,tx3,tx4,tx5,total_eth,probability,weighted_cost_eth
s1,0.000252000,0.000585000,0.000840000,0.000336000,0.001260000,0.003273000,0.200000000,0.000654600
s2,0.000210000,0.000910000,0.000960000,0.000384000,0.001440000,0.003904000,0.200000000,0.000780800
s3,0.000420000,0.001170000,0.001320000,0.000528000,0.001980000,0.005418000,0.200000000,0.001083600
s4,0.000147000,0.000390000,0.001200000,0.000480000,0.001800000,0.004017000,0.200000000,0.000803400
s5,0.000315000,0.000650000,0.000960000,0.000384000,0.001440000,0.003749000,0.200000000,0.000749800


Sum of weighted costs: 0.004072200 ETH


## 7. Why deterministic mean-price and expected-value optimization coincide here

Finite sums can be interchanged, and $x_{it}$ is identical across scenarios:

$$\begin{aligned}
E[C(x)]
&=\sum_s q_s\left(10^{-9}\sum_{i,t}g_i p_{ts}x_{it}\right)\\
&=10^{-9}\sum_{i,t}g_i\left(\sum_s q_s p_{ts}\right)x_{it}\\
&=10^{-9}\sum_{i,t}g_i\bar p_t x_{it}.
\end{aligned}$$

This is equality for **every feasible schedule**, not just equality at one
optimum. The feasible regions also match, so their optimal values and sets of
optimal schedules match. Solver tie-breaking need not select the same member.
Unequal probabilities and shared capacity constraints do not break this proof.
Scenario-adaptive decisions would change the problem and are absent here.

We verify the identity over all $\prod_i d_i=96$ deadline-feasible schedules.
The independent enumeration below uses direct arithmetic, not the production
evaluation functions or a custom branch-and-bound implementation.

In [6]:
enumerated_rows = []
for slots in product(*(range(1, tx.deadline + 1) for tx in transactions)):
    costs = np.array([sum(tx.gas_used * s.prices_gwei[t - 1]
        for tx, t in zip(transactions, slots)) / 1_000_000_000 for s in scenarios])
    mean_objective = sum(tx.gas_used * means[t - 1]
        for tx, t in zip(transactions, slots)) / 1_000_000_000
    enumerated_rows.append({'schedule': slots, 'expected_eth': probabilities @ costs,
        'mean_objective_eth': mean_objective, 'worst_eth': costs.max()})
enumeration = pd.DataFrame(enumerated_rows)
assert len(enumeration) == 96
identity_error = (enumeration.expected_eth - enumeration.mean_objective_eth).abs().max()
assert identity_error < 1e-12
print(f'Checked {len(enumeration)} schedules; largest objective discrepancy = {identity_error:.3e} ETH')
display(enumeration.nsmallest(5, 'expected_eth'))

Checked 96 schedules; largest objective discrepancy = 1.735e-18 ETH


,schedule,expected_eth,mean_objective_eth,worst_eth
82,"(1, 2, 3, 3, 3)",0.004072200,0.004072200,0.005418000
34,"(1, 1, 3, 3, 3)",0.004163200,0.004163200,0.005548000
78,"(1, 2, 3, 2, 3)",0.004197000,0.004197000,0.005754000
74,"(1, 2, 3, 1, 3)",0.004264200,0.004264200,0.005850000
30,"(1, 1, 3, 2, 3)",0.004288000,0.004288000,0.005884000


## 8. CVaR formulation

For a loss distribution, define lower weighted VaR as

$$\operatorname{VaR}_{\alpha}(C)=\inf\{c:\mathbb P(C\le c)\ge\alpha\}.$$

The exact discrete CVaR is

$$\operatorname{CVaR}_{\alpha}(C)
=\min_{\eta\in\mathbb R}\left[\eta+
  \frac{1}{1-\alpha}\sum_s q_s(C_s-\eta)_+\right].$$

Linearize $(C_s-\eta)_+$ using $\xi_s\ge0$ and $\xi_s\ge C_s(x)-\eta$:

$$\min_{x,\eta,\xi}\ \sum_s q_s C_s(x)
 +\lambda\left(\eta+\frac{1}{1-\alpha}\sum_s q_s\xi_s\right).$$

For positive $\lambda$ and positive $q_s$, minimization makes the excess variables
tight. At $\lambda=0$, the auxiliaries are not identified by the objective.
The code therefore omits them in the risk-neutral solve and evaluates the
schedule's true VaR/CVaR independently.

For discrete data a minimizing $\eta$ can be nonunique: with five distinct,
equiprobable costs and $\alpha=0.80$, any threshold between the fourth and fifth
ordered costs minimizes the CVaR expression. Thus `eta_eth` reports the canonical
lower quantile and `solver_eta_eth` separately records the solver's threshold.

Here the worst 20% consists of exactly one complete scenario:

$$\operatorname{CVaR}_{0.80}(C)=\max_s C_s.$$

This equality is specific to the probability/tail configuration, not a general
definition of CVaR. The following calculation uses the risk-neutral schedule.

In [7]:
costs = np.array([expected_value.scenario_costs_eth[s.id] for s in scenarios])
eta, cvar = weighted_var_cvar(costs, probabilities, alpha)
excesses = np.maximum(costs - eta, 0)
display(pd.DataFrame({'cost_eth': costs, 'probability': probabilities,
    'excess_above_eta_eth': excesses,
    'weighted_excess_over_tail_eth': probabilities * excesses / (1 - alpha)},
    index=[s.id for s in scenarios]))
print(f'Canonical VaR: {eta:.9f} ETH')
print(f'CVaR = eta + weighted excess / tail mass = {eta:.9f} + {cvar - eta:.9f} = {cvar:.9f} ETH')
assert isclose(eta, .004017, abs_tol=1e-12)
assert isclose(cvar, costs.max(), abs_tol=1e-12)

,cost_eth,probability,excess_above_eta_eth,weighted_excess_over_tail_eth
s1,0.003273000,0.200000000,0.000000000,0.000000000
s2,0.003904000,0.200000000,0.000000000,0.000000000
s3,0.005418000,0.200000000,0.001401000,0.001401000
s4,0.004017000,0.200000000,0.000000000,0.000000000
s5,0.003749000,0.200000000,0.000000000,0.000000000


Canonical VaR: 0.004017000 ETH
CVaR = eta + weighted excess / tail mass = 0.004017000 + 0.001401000 = 0.005418000 ETH


**Exercise: fractional probability mass.** Suppose costs are
$(0.001,0.002,0.010)$ ETH with probabilities $(0.5,0.4,0.1)$ and $\alpha=0.80$.
What is CVaR? Why would simply averaging all observations with cost at least VaR
give the wrong answer? Predict the value before running the next cell.

In [8]:
# The worst 20% contains all 10% at .010 and only 10% of the atom at .002.
exercise_var, exercise_cvar = weighted_var_cvar([.001, .002, .010], [.5, .4, .1], .8)
manual_tail = (.1 * .010 + .1 * .002) / .2
assert isclose(exercise_cvar, manual_tail, abs_tol=1e-12)
print(f'VaR = {exercise_var:.6f} ETH; CVaR = {exercise_cvar:.6f} ETH')

VaR = 0.002000 ETH; CVaR = 0.006000 ETH


### Why this is a MILP, and what the solver does

All scenario costs, constraints and the auxiliary CVaR objective are linear in
the variables. Assignments are binary and the CVaR auxiliaries are continuous:
this is a mixed-integer linear program. Binary decisions prohibit splitting a
transaction fractionally across slots and create a finite combinatorial choice.

Conceptually, a minimization solver relaxes binaries to $0\le x_{it}\le1$ to
obtain a lower bound. An integer feasible incumbent gives an upper bound. If
needed, branch-and-bound fixes a fractional binary to 0 or 1 in subproblems,
pruning branches whose bounds cannot improve the incumbent. Modern solvers
also use presolve, cuts and heuristics. HiGHS performs this work.

There is useful special structure: the mean-price problem separates by
transaction without capacities. With integer count capacities it is a
capacitated bipartite assignment problem with an integral LP relaxation.
Therefore binary notation alone does **not** mean this risk-neutral toy requires
branching. CVaR couples transactions through scenario portfolio costs and can
remove this integrality property. We do not claim branching actually occurred
without node statistics.

## 9. Risk-aversion experiment

Fix $\alpha=0.80$ and vary $\lambda\in\{0,0.1,0.25,0.5,1\}$. A 95% tail with
only five scenarios gives no credible statistical resolution: its empirical
CVaR also equals the maximum. The future real-data experiment may use
$\alpha=0.95$ with many more scenarios and enough effective tail observations.

Costs of different schedules are evaluated on the **same** five trajectories.
The objective's risk penalty is a preference term, not an extra on-chain bill.

In [9]:
sensitivity = sensitivity_table()
display(sensitivity)

# Independent oracle for this particular equal-probability alpha=.80 setting.
for row in sensitivity.itertuples(index=False):
    brute_objective = (enumeration.expected_eth + row.lambda_risk * enumeration.worst_eth).min()
    assert isclose(row.objective_value_eth, brute_objective, abs_tol=1e-10)
    assert row.solver_status == 'optimal'
    assert isclose(row.cvar_eth, row.worst_case_cost_eth, abs_tol=1e-12)
print('All five HiGHS objective values match exhaustive enumeration.')

,lambda_risk,expected_cost_eth,cvar_eth,worst_case_cost_eth,eta_eth,objective_value_eth,schedule_tx1_to_tx5,solver_status
0,0.000000000,0.004072200,0.005418000,0.005418000,0.004017000,0.004072200,"(1, 2, 3, 3, 3)",optimal
1,0.100000000,0.004072200,0.005418000,0.005418000,0.004017000,0.004614000,"(1, 2, 3, 3, 3)",optimal
2,0.250000000,0.004072200,0.005418000,0.005418000,0.004017000,0.005426700,"(1, 2, 3, 3, 3)",optimal
3,0.500000000,0.004072200,0.005418000,0.005418000,0.004017000,0.006781200,"(1, 2, 3, 3, 3)",optimal
4,1.000000000,0.004396200,0.005058000,0.005058000,0.004917000,0.009454200,"(1, 2, 3, 3, 4)",optimal


All five HiGHS objective values match exhaustive enumeration.


## 10. Results

The risk-neutral solution and the solutions through $\lambda=0.5$ use
$(1,2,3,3,3)$. At $\lambda=1$, transaction 5 moves to slot 4:
$(1,2,3,3,4)$. We compare each scenario, the canonical VaR, and the solver's
threshold. A change in solver threshold alone is not evidence of a changed tail.

In [10]:
risk_averse = solve_stochastic(transactions, scenarios, OptimizationConfig(alpha=alpha, lambda_risk=1))
comparison = pd.DataFrame({'risk_neutral_eth': expected_value.scenario_costs_eth,
                           'risk_averse_eth': risk_averse.scenario_costs_eth})
comparison['change_eth'] = comparison.risk_averse_eth - comparison.risk_neutral_eth
display(comparison)
display(pd.DataFrame([
    {'lambda': risk, 'expected_eth': result.expected_cost_eth, 'canonical_var_eth': result.eta_eth,
     'solver_eta_eth': result.solver_eta_eth, 'independent_cvar_eth': result.cvar_eth,
     'solver_cvar_eth': result.solver_cvar_eth}
    for risk, result in [(0, expected_value), (1, risk_averse)]
]))

,risk_neutral_eth,risk_averse_eth,change_eth
s1,0.003273000,0.003993000,0.000720000
s2,0.003904000,0.003544000,-0.000360000
s3,0.005418000,0.005058000,-0.000360000
s4,0.004017000,0.004917000,0.000900000
s5,0.003749000,0.004469000,0.000720000


,lambda,expected_eth,canonical_var_eth,solver_eta_eth,independent_cvar_eth,solver_cvar_eth
0,0,0.004072200,0.004017000,NaN,0.005418000,NaN
1,1,0.004396200,0.004917000,0.005058000,0.005058000,0.005058000


In [11]:
diagnostics = pd.DataFrame([
    {'model': label, 'variables': result.num_variables, 'binary_variables': result.num_binary_variables,
     'constraints': result.num_constraints, 'termination': result.solver_status,
     'objective_eth': result.objective_value_eth, 'lower_bound_eth': result.objective_bound_eth,
     'relative_gap': result.optimality_gap, 'node_count': result.node_count}
    for label, result in [('Mean prices', deterministic), ('Expected scenarios', expected_value),
                          ('Mean + CVaR (lambda=1)', risk_averse)]
]).set_index('model')
display(diagnostics)
print('Counts are before presolve. Node count is unavailable through the public APPSI result.')

# Formulations remain inspectable as ordinary Pyomo models.
risk_model = build_stochastic_model(transactions, scenarios, OptimizationConfig(lambda_risk=1))
print('tx2 assignment:', risk_model.assign_once['tx2'].expr)
print('Scenario s1 cost:', risk_model.scenario_cost['s1'].expr)
print('Scenario s1 CVaR constraint:', risk_model.excess['s1'].expr)
assert all(v.is_binary() for v in risk_model.x.values())

,variables,binary_variables,constraints,termination,objective_eth,lower_bound_eth,relative_gap,node_count
model,,,,,,,,
Mean prices,14,14,5,optimal,0.004072200,0.004072200,0.000000000,None
Expected scenarios,14,14,5,optimal,0.004072200,0.004072200,0.000000000,None
Mean + CVaR (lambda=1),20,14,10,optimal,0.009454200,0.009454200,0.000000000,None


Counts are before presolve. Node count is unavailable through the public APPSI result.
tx2 assignment: x[tx2,1] + x[tx2,2]  ==  1
Scenario s1 cost: 1e-09*(252000*x[tx1,1] + 780000*x[tx2,1] + 585000*x[tx2,2] + 1440000*x[tx3,1] + 1080000*x[tx3,2] + 840000*x[tx3,3] + 1320000*x[tx3,4] + 576000*x[tx4,1] + 432000*x[tx4,2] + 336000*x[tx4,3] + 2160000*x[tx5,1] + 1620000*x[tx5,2] + 1260000*x[tx5,3] + 1980000*x[tx5,4])
Scenario s1 CVaR constraint: (1e-09*(252000*x[tx1,1] + 780000*x[tx2,1] + 585000*x[tx2,2] + 1440000*x[tx3,1] + 1080000*x[tx3,2] + 840000*x[tx3,3] + 1320000*x[tx3,4] + 576000*x[tx4,1] + 432000*x[tx4,2] + 336000*x[tx4,3] + 2160000*x[tx5,1] + 1620000*x[tx5,2] + 1260000*x[tx5,3] + 1980000*x[tx5,4])) - eta  <=  xi[s1]


## 11. Interpretation

Moving transaction 5 to slot 4 changes scenario prices by $(4,-2,-2,5,4)$
gwei/gas. Multiplying by 180,000 gas and $10^{-9}$ gives scenario cost changes
$(0.00072,-0.00036,-0.00036,0.00090,0.00072)$ ETH. The formerly worst scenario
improves, while some other scenarios become more expensive.

Expected cost rises by **0.000324 ETH** (about 7.96%), while CVaR/worst cost falls
by **0.000360 ETH** (about 6.64%). Interestingly, VaR rises even as CVaR falls:
these measure different features of the loss distribution.

Decisions are discrete, so sensitivity can be flat over intervals. Enumerating
the feasible expected-cost/worst-cost pairs reveals three nondominated schedules.

In [12]:
frontier_rows = []
for row in enumeration.itertuples(index=False):
    weakly_better = (enumeration.expected_eth <= row.expected_eth + 1e-12) & (enumeration.worst_eth <= row.worst_eth + 1e-12)
    strictly_better = (enumeration.expected_eth < row.expected_eth - 1e-12) | (enumeration.worst_eth < row.worst_eth - 1e-12)
    if not (weakly_better & strictly_better).any():
        frontier_rows.append({'schedule': row.schedule, 'expected_eth': row.expected_eth, 'cvar_eth': row.worst_eth})
frontier = pd.DataFrame(frontier_rows).sort_values('expected_eth')
display(frontier)
expected_increase = risk_averse.expected_cost_eth - expected_value.expected_cost_eth
tail_reduction = expected_value.cvar_eth - risk_averse.cvar_eth
break_even_lambda = expected_increase / tail_reduction
print(f'Expected-cost increase: {expected_increase:.9f} ETH ({expected_increase / expected_value.expected_cost_eth:.2%})')
print(f'CVaR reduction:         {tail_reduction:.9f} ETH ({tail_reduction / expected_value.cvar_eth:.2%})')
print(f'Break-even lambda: {break_even_lambda:.6f}')
assert isclose(break_even_lambda, .9, abs_tol=1e-10)
tie_objectives = enumeration.expected_eth + .9 * enumeration.worst_eth
ties = enumeration[np.isclose(tie_objectives, tie_objectives.min(), atol=1e-12, rtol=0)]
assert len(ties) == 3
display(ties[['schedule', 'expected_eth', 'worst_eth']])

,schedule,expected_eth,cvar_eth
0,"(1, 2, 3, 3, 3)",0.004072200,0.005418000
2,"(1, 2, 4, 3, 3)",0.004288200,0.005178000
1,"(1, 2, 3, 3, 4)",0.004396200,0.005058000


Expected-cost increase: 0.000324000 ETH (7.96%)
CVaR reduction:         0.000360000 ETH (6.64%)
Break-even lambda: 0.900000


,schedule,expected_eth,worst_eth
82,"(1, 2, 3, 3, 3)",0.004072200,0.005418000
83,"(1, 2, 3, 3, 4)",0.004396200,0.005058000
94,"(1, 2, 4, 3, 3)",0.004288200,0.005178000


The schedules $(1,2,3,3,3)$, $(1,2,4,3,3)$ and $(1,2,3,3,4)$ tie at
$\lambda=0.9$. At that exact weight, different optimal schedules are legitimate.
Below the threshold the cheaper schedule wins; above it, among the nondominated
choices, the schedule with the smallest worst-case cost wins.

**Optional Phase 1 extension:** rerun with `slot_capacities=(2, 1, 1, 2)` and
compare the assignment and risk tradeoff. Capacity is a business constraint on
transaction counts, not Ethereum block gas capacity. The test suite already
checks feasibility and exhaustive optimality under this example capacity.

## 12. Limitations

- This is in-sample synthetic validation, not an empirical savings estimate or
  an out-of-sample performance claim. Five scenarios cannot establish tail-risk
  accuracy. No inferential confidence intervals are warranted.
- The schedule is fixed before any price observation. There are no adaptive
  decisions, forecasts, transaction dependencies, nonce ordering or recourse.
- Deadlines, availability and optional capacities are business assumptions;
  historical blockchain records do not reveal a treasury's willingness to wait.
- Gas use is deterministic. Transactions are price-taking and assumed to execute
  successfully in their assigned slots. Base fees, tips, fee caps, failed
  transactions and inclusion uncertainty are not modeled separately.
- ETH is the only reporting currency. There is no ETH/USD conversion or FX risk.
- The special CVaR=max identity should not be extrapolated to a larger empirical
  distribution. Our metrics and tests also cover fractional probability atoms.
- The solver's reported optimality concerns this mathematical model and its
  tolerances, not the realism of these assumptions. No custom solver is used.

## 13. Next step: real Ethereum data

**Planning only: Phase 2 requires separate authorization.** No data connection
or pipeline is implemented in this deliverable.

The next study should define slot duration, align whole historical trajectories,
and choose a consistent Mainnet price measure. A base fee alone is not an
all-in effective gas price. BigQuery Blockchain Analytics or Dune are candidate
providers for historical timestamps, fee fields and real transaction `gas_used`.
Deadlines and priorities will remain explicit business assumptions.

To test economic performance, estimate schedules using historical training
windows and evaluate them on later, untouched trajectories. Avoid overlapping
windows across the training/test boundary and future-information leakage.
Compare immediate execution, mean-price scheduling and CVaR scheduling; report
the expected-value formulation's equivalence rather than inventing a separate
advantage. Use many more scenarios before considering $\alpha=0.95$.

React, FastAPI and Docker are later demonstration tools, after the mathematical
and empirical study. They are intentionally absent from Phase 1.

Implementation reference: [Pyomo APPSI result and termination documentation](https://pyomo.readthedocs.io/en/6.9.3/api/pyomo.contrib.appsi.base.Results.html).
All numerical results above are computed locally from the supplied synthetic data.